# STXM Analysis Optimized

**Context**: Materials Science PhD Data Analysis (Battery Materials)  
**Purpose**: High-throughput processing of STXM data (Alignment, Denoising, Background Removal, Saturation Correction).  
**Core Library**: `stxm_utils.TXMAnalyzer`

In [ ]:
import sys
from pathlib import Path

# Add repository root to sys.path
repo_root = Path.cwd().parent.parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

# Add current directory (for stxm_utils)
if str(Path.cwd()) not in sys.path:
    sys.path.append(str(Path.cwd()))

In [ ]:
import matplotlib.pyplot as plt
import optuna
from stxm_utils import TXMAnalyzer

# Import Project Modules
from Figure.config import OUTPUT_ROOT, setup

# Setup plotting style
colors = setup()

path_out = OUTPUT_ROOT
print(f"Output Path: {path_out}")

In [ ]:
# Target Parameters
target_element = "Mn"
filename = "E1A_specnorm_aliOF.hdf5.hdf5"

# Initialize Analyzer with simplified path handling
analyzer = TXMAnalyzer(filename, element=target_element, lazy=False)
print(f"Raw data loaded from: {analyzer.file_path}")
print(f"Raw data shape: {analyzer.raw_signal.data.shape}")

In [ ]:
# --- OPTIONAL: REDUCE DATA SIZE ---
print(f"Original Shape: {analyzer.raw_signal.data.shape}")

# CROP to 50x50 pixels spatial area for faster testing
analyzer.raw_signal = analyzer.raw_signal.isig[:50, :50]

print(f"Reduced Shape: {analyzer.raw_signal.data.shape}")

In [ ]:
# Preprocess Data
analyzer.preprocess(new_energy_scale=0.1)

# Validate Data Shape
print(f"Axes: {analyzer.od_signal.axes_manager}")

# Quick Visualization of Sum Image
# Use constrained_layout for better spacing and project-standard colormap
fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
im = ax.imshow(analyzer.od_signal.sum(axis=-1).data, cmap="sunset", origin="lower")
ax.set_title("Sum Image (Optical Density)")
ax.axis("off")
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Intensity")
plt.show()

In [ ]:
# PCA Denoising
analyzer.denoise(components_number=None)  # Auto-detect via elbow method
print("PCA Denoising complete.")

In [ ]:
# Remove Background using element-specific parameters
analyzer.remove_background(element=target_element, component_name="Offset")

# Compare Spectra (Before vs After)
s_raw = analyzer.od_signal.mean((0, 1))
s_clean = analyzer.denoised_signal.mean((0, 1))

# Use constrained_layout and consistent styling
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
ax.plot(s_raw.axes_manager[0].axis, s_raw.data, label="Raw OD", color=colors[0], linewidth=1.5, alpha=0.7)
ax.plot(s_clean.axes_manager[0].axis, s_clean.data, label="Bg Removed + PCA", color=colors[1], linewidth=1.5)
ax.legend(frameon=True)
ax.set_xlabel("Energy (eV)")
ax.set_ylabel("Optical Density")
ax.set_title("Average Spectrum")
ax.grid(True, linestyle="--", alpha=0.5)
plt.show()

In [ ]:
# Apply B-value correction using TXMAnalyzer method with automatic thresholding
print("Running B-value optimization (this may take a moment)...")
best_k, best_popt, best_r2 = analyzer.apply_b_value_correction(element=target_element, num_points=50)

print("\nOptimization Result:")
print(f"  K (Threshold): {best_k:.4f}")
print(f"  R² (Fit Quality): {best_r2:.4f}")
print(f"  Kappa (Stray Light): {best_popt[0]:.4f}")

In [ ]:
# Define ROI - 'full' means whole image
analyzer.set_roi("full", element=target_element)

# Perform Standard KMeans Clustering
analyzer.perform_clustering(rois="full", n_clusters=3, algorithm="kmeans", exclude_first_component=True)

# Visualization using new utility method
analyzer.plot_clusters("full")

### Automatic Parameter Optimization
Use **Optuna** to find the best UMAP and HDBSCAN parameters that maximize chemical separation (Silhouette Score) and minimize noise.

In [ ]:
# 1. Run Optimization
study = analyzer.optimize_umap_params(
    rois="full",
    n_trials=50,
    exclude_first_component=True
)

print("Best Params Found:", study.best_params)

# 2. Visualize Optimization History
fig = optuna.visualization.plot_optimization_history(study)
fig.update_layout(height=400, title="Optimization History")
fig.show()

# 3. Visualize Parameter Importance
fig2 = optuna.visualization.plot_param_importances(study)
fig2.update_layout(height=400, title="Parameter Importance")
fig2.show()

In [ ]:
# 3. Apply Best Parameters
# We unpack the best_params dictionary directly into the function
analyzer.perform_umap_clustering(
    rois="full",
    exclude_first_component=True,
    # UMAP Params
    n_neighbors=study.best_params["n_neighbors"],
    n_components=study.best_params["n_components"],
    min_dist=study.best_params["min_dist"],
    # HDBSCAN Params
    min_cluster_size=study.best_params["min_cluster_size"],
    # Optional: map min_samples if it was optimized, else default
    # min_samples=study.best_params.get("min_samples", None)
)

# Plot Clusters
analyzer.plot_clusters("full")

In [ ]:
# Define Model Configuration for Mn L-edge using DoubleStep component
peak_configs = [
    {"type": "Gaussian", "name": "L3", "param_guesses": {"centre": 640.0, "sigma": 1.0, "height": 1.0}, "fit_range": (635.0, 645.0)},
    {"type": "Gaussian", "name": "L2", "param_guesses": {"centre": 651.0, "sigma": 1.0, "height": 0.5}, "fit_range": (648.0, 655.0)},
    {"type": "DoubleStep", "name": "Step", "element": target_element},
]

print("Starting Model Fitting (Pixel-by-Pixel)... using fit_global_first for stability.")

# Fit Model with global pre-fit
analyzer.fit(peak_configs=peak_configs, rois="full", optimizer="lm", freeze_strategy="final_only", fit_global_first=True)

print("Fitting complete.")

In [ ]:
# Visualization of Parameter Maps using new utility method
analyzer.plot_fit_results("full")

In [ ]:
# Export all results to xarray DataTree for archival and complex downstream analysis
dt = analyzer.export_to_datatree()

# Save corrected signal to Hspy format
s_corrected = analyzer.corrected_signal if analyzer.corrected_signal is not None else analyzer.denoised_signal
save_filename = f"{analyzer.file_path.stem}_Corrected.hspy"
s_corrected.save(path_out / save_filename, overwrite=True)
print(f"Data saved to: {path_out / save_filename}")

# Display final DataTree structure
dt